# Pull Statcast pitch data (2021–2025)

Run this **on your desktop** (Baseball Savant must be reachable — it's blocked in locked-down cloud boxes).

One row per pitch, all columns. Saves one parquet per season to `data/raw/`. Resumable and cached.

**Expect:** ~700–750k pitches/season · ~180–230 MB/season as parquet · ~1 GB for 5 seasons. Cold pull ~10–25 min/season (many small daily requests); near-instant on re-run thanks to the cache.

In [ ]:
# 1. Install dependencies (once)
%pip install "pybaseball>=2.2.7" "pandas>=2.0" "pyarrow>=14.0"

In [ ]:
# 2. Smoke test — pull a single day first to confirm Savant is reachable and see the columns
from pybaseball import statcast, cache
cache.enable()

sample = statcast(start_dt="2024-04-01", end_dt="2024-04-01")
print(f"{len(sample):,} pitches, {sample.shape[1]} columns")
print("delta_run_exp populated:", f"{sample['delta_run_exp'].notna().mean():.1%}")
sorted(sample.columns.tolist())

In [ ]:
# 3. Full pull, one season at a time (resumable: skips seasons already saved)
import os, time

SEASONS = range(2021, 2026)   # 2021–2025 inclusive; widen to range(2015, 2026) for the long history
OUTDIR = "data/raw"
os.makedirs(OUTDIR, exist_ok=True)

for yr in SEASONS:
    out = f"{OUTDIR}/statcast_{yr}.parquet"
    if os.path.exists(out):
        print(f"[{yr}] already saved ({os.path.getsize(out)/1e6:,.0f} MB) — skipping")
        continue
    print(f"[{yr}] pulling — be patient (many daily requests)...")
    t0 = time.time()
    df = statcast(start_dt=f"{yr}-03-15", end_dt=f"{yr}-11-30")
    df.to_parquet(out, index=False)
    print(f"[{yr}] {len(df):,} pitches → {out} "
          f"({os.path.getsize(out)/1e6:,.0f} MB, {(time.time()-t0)/60:.1f} min)")

In [ ]:
# 4. (Optional) Build a combined SQLite db that matches the pitch-sequencing repo's ingest
import glob, sqlite3, pandas as pd

db = f"{OUTDIR}/statcast.db"
if os.path.exists(db):
    os.remove(db)
con = sqlite3.connect(db)
total = 0
for p in sorted(glob.glob(f"{OUTDIR}/statcast_*.parquet")):
    d = pd.read_parquet(p)
    d.to_sql("statcast", con, if_exists="append", index=False)
    total += len(d)
con.execute("CREATE INDEX IF NOT EXISTS idx_ab ON statcast(game_pk, at_bat_number, pitch_number)")
con.commit(); con.close()
print(f"{total:,} pitches → {db} ({os.path.getsize(db)/1e6:,.0f} MB)")

In [ ]:
# 5. Verify — load it back and sanity-check
import glob, pandas as pd
files = sorted(glob.glob(f"{OUTDIR}/statcast_*.parquet"))
df = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)
print(f"{len(df):,} pitches across {df['game_date'].str[:4].nunique() if df['game_date'].dtype==object else df['game_date'].dt.year.nunique()} seasons")
print("\nby game_type:")
print(df['game_type'].value_counts())
print("\ntop pitch types:")
print(df['pitch_type'].value_counts().head(10))
print("\ndelta_run_exp populated:", f"{df['delta_run_exp'].notna().mean():.1%}")